# E54: Does the decomposition tell a half-known word map from a cipher where steering cannot?

## The answer

Yes for the decomposition, and only partly for steering. We fine-tuned the small model on a word table it already answers weakly and on an arbitrary cipher over the same keys. Switching existing subcomponents on carried about three quarters of the first table's gain and about a quarter of the cipher's. One added steering vector moved both tasks by about the same amount. The decomposition separates the pair more clearly than steering in every comparison we can make, by a margin that depends on the units. Our steering prediction failed.

## What we were trying to test

The project asks whether a VPD decomposition can tell a fine-tune that strengthens something the model already does from one that teaches something new. An earlier run found that one steering vector separates a copy task from an arbitrary map as well as the decomposition does. Copying is the same for every key, so a single added vector can carry it, and the map is not. That made the earlier comparison too easy for steering.

Here both tasks depend on the key. Each window is Pile text with a rare marker token, then a key word; the model must predict the key's value next. The two tasks share 32 keys and the same windows, and differ only in the table.

- **Elicit map.** Each key's value is a word the original model already ranks 2nd to 20th after the marker and key. Examples are " think" → " it" and " action" → " potential". The model's top guess is excluded, so a vector that only sharpens its first guess cannot get the answer.
- **Cipher.** The same keys mapped to words the original model almost never predicts there.

Both are fine-tuned on all 24 decomposed matrices with the earlier recipe, two seeds each. The elicit map learns by step 10. The cipher learns more slowly, so it is read at the step whose weight change has the same size as the elicit map's, step 9. It is also read at its own scored step, 30, to check that the result does not hang on where it was stopped.

Each state is read two ways. Both read 0 for the original model and 1 for the fine-tuned model.

- **Re-gating share** $r$: keep every subcomponent at its original weights and let only the causal-importance function Γ see the fine-tuned model's activations. The share of the fine-tune's log-odds gain this recovers is $r$. High $r$ means the fine-tune works by switching existing subcomponents on.
- **Steering share** $S$: add to the original model, at the answer position, the mean difference between the fine-tuned and original residual streams. The share of the fine-tune's gain this recovers is $S$, scored on windows not used to fit the vector.

A pair separates when the gap between the two tasks' means beats the larger spread between seeds and is at least 0.2.

## What we expected, and why

- **The decomposition separates the pair, elicit higher.** A half-known table should be reached by switching on subcomponents that already produce those words. A cipher needs changed subcomponents.
- **Steering does not separate them, and stays at or under 0.3 on both.** One mean vector is the same for every key. So it should carry little of either map. We noted one way it could still work: lowering a rival word shared by many keys.
- **The cipher's re-gating share barely moves between the matched step and its own step,** by under 0.1. Otherwise the comparison would depend on update size.

Before the fine-tunes, the job re-read two saved states from the earlier run. Both readings reproduced the earlier numbers to four decimals.

## Does the decomposition read the elicit map higher than the cipher?

In [ ]:
from pathlib import Path
from IPython.display import Image, display
FIG = Path("../experiments/keydependent_elicit/figures")
display(Image(filename=str(FIG / "p1_route_split.png")))

**Reading it.** Yes, by a wide margin. Switching existing subcomponents on, with none changed, recovers 0.77 and 0.76 of the elicit map's gain. It recovers 0.24 and 0.21 of the matched cipher's. The gap is 0.54, against a seed spread of 0.03. The new pair lands where the earlier copy and cipher pair did, on the left. Copy and the elicit map differ in whether the answer depends on the key, yet they read alike. One caveat: the matched cipher is only part-learned, with its target at probability 0.23. Its log-odds gain, the denominator of this share, still matches the elicit map's closely.

## Does one steering vector fail to separate the pair?

In [ ]:
display(Image(filename=str(FIG / "p2_steering.png")))

**Reading it.** The prediction failed. One mean vector recovers 0.38 and 0.39 of the elicit map's gain, above the filed cap of 0.3 drawn in green. On the matched cipher it recovers 0.23 and 0.19. The gap is 0.18, under the filed minimum of 0.2, so steering does not separate the pair by our rule. It separates this pair far less than it separated copy from cipher on the left.

Both halves of the miss depend on choices in the measurement:

- **The cap miss comes mostly from the unit.** The share is of a gain in nats, and the elicit map's nats gain is compressed because its target ends near probability 0.96. In absolute terms the vector moves both tasks about equally. It recovers 2.54 and 2.62 nats on the elicit map. On the matched cipher it recovers 2.43 and 1.93.
- **"Does not separate" depends on the matched cipher being part-learned,** with its target at probability 0.23. At the cipher's own step, where it is learned, its steering share is 0.11 and 0.07. There the gap is 0.30, and steering would separate the pair by the filed rule.

The filed verdict stays as scored at matched size, the user's choice, recorded in `decisions.md`. The decomposition separates the pair more clearly than steering does. One mean vector moves a real part of the elicit map without separating the pair by the filed bar.

How much more clearly depends on units, because the steering share is of a nats gain and the re-gating share of a log-odds gain. The ratio of the two gaps runs from about 1.7 at the cipher's own step to over 10 with steering estimated in log-odds. Measured against each reading's noise floor it is about 6. The direction holds in every case.

## Does the cipher's re-gating share depend on how far it was trained?

In [ ]:
display(Image(filename=str(FIG / "p3_size.png")))

**Reading it.** No, within this range. From step 9 to step 30 the cipher's weight change nearly doubles. Its re-gating share moves by only 0.02 to 0.03, well inside the shaded band of 0.1. So the matched comparison above is not an artefact of where the cipher was stopped. Steering behaves differently: the cipher's steering share roughly halves over the same steps, to 0.11 and 0.07.

## What we learned

On a pair built so that one vector should fail on both tasks, the re-gating share still separates a half-known map from a cipher. So on the earlier copy and cipher pair it was not reading only whether the answer depends on the key. The steering vector moves both tasks by about the same amount, near 2.5 nats. Nothing here shows it carries anything specific to the elicit keys; that is an unmeasured hypothesis. The supported claim is a direction: the decomposition separates this pair more clearly than one mean vector, by a margin from under twofold to over tenfold.

A limit the user named: a new circuit built from existing subcomponents also reads as "elicited" to this measure. At the resolution of subcomponents, "elicited" means "built from old parts", not "an old behaviour".

## How compelling is the evidence

- Two seeds per task, one small model, one task format. The seed spread is a range, not a variance.
- The elicit label rests on the original model's behaviour, not on a check of its internal representations. On the selection windows, per-key probabilities span more than three orders of magnitude, so some keys are barely half-known.
- The stage that picked the table first failed its own gate. The original model's probability of the elicit targets, averaged geometrically over held-out windows, read 1.7e-3 against a filed floor of 5e-3. Values had been chosen by the plain average, which reads higher. The user lowered the floor to 1e-3 **after seeing the value**, before any fine-tune, with the same keys and values. It is a dated amendment in the plan.
- The startup check reproduces earlier numbers; it is not a positive control of known kind for either reading.
- The margin between the two readings was first an estimate across units; Amendment 1 below measured it in one unit, at about tenfold.

## Surprises

**The steering prediction failed.** We predicted one mean vector would recover at most 0.3 of either task. It recovered 0.38 and 0.39 of the elicit map. The pair still does not separate under steering by the filed bar. There are two candidate readings, neither measured. About half the elicit values are common function words such as " of" and " the", which one vector can raise together. And the same nats make a larger share of the elicit map's smaller gain.

**More training halves the cipher's steering share but leaves its re-gating share in place.** At step 30 the vector recovers fewer nats than at step 9.

## What next

Labelled provisional, because the steering prediction failed; the user chose this line.

- **The same pair on Qwen.** If the decomposition still separates and steering separates less, the narrow claim holds beyond the small model.
- **Jain et al.'s readout check on the elicit keys.** Read each key's value linearly from the original model's activations and test it causally. If it fails on many keys, the elicit label is weaker than the behaviour suggests.

## Departures from the literature

- **Steering.** Contrastive activation addition, Rimsky et al. 2024, contrasts prompts within one model, adds the vector at every later position and sweeps a multiplier. Ward et al. 2025 and Venhoff et al. 2025 steer reasoning models and judge generated text. We take the tuned-minus-original difference on the same inputs, add it once at the answer position with multiplier 1, and read one token's log probability. The cap of 0.3 and the minimum gap are ours.
- **The re-gating share is not in the VPD paper.** VPD defines subcomponents, Γ and the CI-masked model. Splitting a later fine-tune's gain into parts is this project's addition.
- **Jain et al. 2024 define possessing a capability by a linear readout.** Our elicit label uses the original model's next-token probability instead. The readout was not run.
- **The fine-tune recipe and step rule are this project's own,** from the earlier run, not a cited recipe. They are reused unchanged so the new states sit beside the old ones.
- **The two readings use different units,** nats for steering and log-odds for the re-gating share. That is an internal difference, not a departure from a source, but it sets how large the margin between them looks.

## Scorecard

| id | claim | read | verdict |
|---|---|---|---|
| S0 | controls reproduce earlier r and S | r 0.8024, 0.2733; S 0.7503, 0.1583 | pass |
| S1 | elicit rate in band, cipher under 1e-4 | 1.70e-3, floor amended to 1e-3 after seeing it; 1.08e-6 | pass at the amended floor |
| P1 | r separates, elicit higher | 0.77, 0.76 vs 0.24, 0.21; gap 0.54 | pass |
| P2 | S does not separate, all ≤ 0.3 | 0.38, 0.39 vs 0.23, 0.19; gap 0.18; elicit over 0.3 | **fails** |
| P3 | cipher's r within 0.1 across steps | 0.024, 0.031 | pass |
| P4 | prompting separates | no working prompt format | not run |

## Not run

- Prompting the original model as a label check; no working prompt format exists for this model.
- Jain et al.'s readout and causal check on the elicit keys.
- Neuron patching and weight statistics on the new pair.
- A random update of matched size as a null.
- The copy task on these keys; a cipher matched on learning progress rather than size.
- Steering read in log-odds units on the saved states: since run, in Amendment 1.
- Steering at the cipher's own step scored as a separation: since run, in Amendment 1.
- More than two seeds; per-key breakdowns; which subcomponents switch on.
- Qwen.

## Provenance

Full record: `reports/analyses/2026-10-30_analysis_e54-keydependent-elicit.md`. Results: `experiments/keydependent_elicit/results/stage0.json`, `stage1.json`, `stage2.json`; rules in `predictions.json`. Run on a MATS cluster L40 in the debug queue, at no cost. Job 45963 stopped at the filed floor, job 45973 reran the first two stages, and job 45974 ran the fine-tunes and readings. Figures from `plot.py`.

## Amendment 1: read in the re-gating share's unit, does steering still fail to separate the pair?

We re-read the six saved weight changes and measured steering's share of each fine-tune's gain in log-odds, the unit the re-gating share uses. In that unit steering recovers about a quarter of the elicit map's gain and about a fifth of the matched cipher's, so it still does not separate the pair. The decomposition's gap between the two tasks is now measured at about ten times steering's, and the answer at the top stands.

**What was measured.** The steering vector, its block and the 48 scoring windows stay as in the main run; only the readout changes. The new share's numerator is the rise in the target's mean log-odds when the vector is added to the original model. Its denominator is the fine-tune's rise in mean log-odds on the same windows. It reads 0 for the original model and 1 for the fine-tuned model, like the re-gating share. Before any log-odds reading, the job recomputed the nats share on all six saved states through the new code path. It matched the main run to every printed digit, and every saved file's hash matched.

**What we expected.** About 0.25 on the elicit map and 0.22 and 0.18 on the matched cipher, from dividing the vector's nats by each state's log-odds gain. So a gap near 0.05, and a ratio of the two readings' gaps of over tenfold.

In [ ]:
display(Image(filename=str(FIG / "a1_units.png")))

**Reading it.** Steering still does not separate the pair in the re-gating share's unit. On the elicit map it recovers 0.246 and 0.252 of the log-odds gain. On the matched cipher it recovers 0.219 and 0.175. That gap of 0.052 is just above the larger seed range of 0.043 and far under the filed minimum of 0.2. The re-gating share's gap on the same states is 0.54, so its gap is 10.3 times steering's. That replaces the estimate of over tenfold; the measured steering gap is a little larger than the estimated 0.04. The ratio is noisy, because steering's gap is barely above its seed range: one range wider and it would read about 6.

The unit also explains the earlier cap miss. In nats the elicit map's steering share was 0.38 and 0.39, above the filed cap of 0.3. In log-odds it is 0.25 on both seeds. The filed prediction stays failed as scored; this is the same vector read in another unit, after the fact.

**Two checks the code review asked for.** First, the log-odds readout had no check on the GPU before scoring. The run printed one instead of stopping on it. The fine-tune's log-odds gain read through the steering code and through the route split agrees within 1.4e-4 to 4.7e-4 relative on all six states. Second, the cipher's-own-step score read the main run's nats share, not the one the startup check recomputes. Here the two are identical, so the score is the same.

| id | claim | read | verdict |
|---|---|---|---|
| stage 0 | saved states and new code path reproduce the main run's S | all six, difference 0 | pass |
| P1-lo | steering in log-odds does not separate the pair | 0.246, 0.252 vs 0.219, 0.175; gap 0.052 | pass |
| P2-lo | re-gating share minus steering in log-odds ≥ 0.3 on elicit | 0.526, 0.506 | pass |
| P3-own | steering in nats separates elicit from cipher at its own step | gap 0.296, range 0.048 | pass, not a test |

**Departures from the literature.** The steering recipe departs from contrastive activation addition as listed above. Reading its effect as a share of a log-odds gain is this project's choice, to match the route split. The thresholds of 0.2 and 0.3 are ours.

**Not run.** The steering share in log-odds with its block chosen in log-odds; at more than one position or with a multiplier; per key; a random update of matched size as a null. Full record: the Amendment 1 section of the analysis report; results in `experiments/keydependent_elicit/results/amend1.json`, job 46070, at no cost.